In [ ]:
from rp import *
from dataset import EnvatoDataset
git_import('CommonSource')
import rp.git.CommonSource.noise_warp as nw
device=select_torch_device(prefer_used=True, reserve=True)

In [ ]:
new_dataset_root = '/efs/users/jordanlin/public/ryan/CleanCode/Datasets/Envato/Noisewarp/480P-4x81'
frame_stride = 4
video_length = 81
height = 480
width = 832

In [ ]:
dataset = EnvatoDataset(num_frames=video_length*frame_stride)

@globalize_locals
def load_random_video():
    while True:
        try:
            index =random_index(dataset)
            video, prompt = dataset[index]
            break
        except Exception as e:
            fansi_print(f'load_random_video: {type(e).__name__} at index={index}: {e}', 'red')
            # print_stack_trace()
            
    video = resize_list(video, video_length * frame_stride) #Just in case it's too slow...
    video = video[::frame_stride]
    video = resize_images_to_hold(video, height=height, width=width)
    video = crop_images(video, height=height, width=width, origin='center')
    video = as_numpy_array(video)
    video = as_byte_images(video)
    
    sample_root = path_join(new_dataset_root, f'{index}')
    video_path = path_join(sample_root, 'video.mp4')
    prompt_path = path_join(sample_root, 'prompt.txt')
    noise_folder = path_join(sample_root, 'noisewarp')
    display_dict(gather_vars('index prompt video_path prompt_path'))
    
    save_video_mp4(video, video_path, video_bitrate='high', framerate=60, show_progress=False)
    save_text_file(prompt, prompt_path)

load_random_video()

In [ ]:
@globalize_locals
def do_noisewarp():
    FRAME = 2**-1 #We immediately resize the input frames by this factor, before calculating optical flow
                  #The flow is calulated at (input size) × FRAME resolution.
                  #Higher FLOW values result in slower optical flow calculation and higher intermediate noise resolution
                  #Larger is not always better - watch the preview in Jupyter to see if it looks good!
    FLOW = 2**3   #Then, we use bilinear interpolation to upscale the flow by this factor
                  #We warp the noise at (input size) × FRAME × FLOW resolution
                  #The noise is then downsampled back to (input size)
                  #Higher FLOW values result in more temporally consistent noise warping at the cost of higher VRAM usage and slower inference time
    LATENT = 8    #We further downsample the outputs by this amount - because 8 pixels wide corresponds to one latent wide in Stable Diffusion
                  #The final output size is (input size) ÷ LATENT regardless of FRAME and FLOW
    
    LATENT = 8    #Uncomment this line for a prettier visualization! But for any Stable-Diffusion based model, use LATENT=8
    
    #See this function's docstring for more information!
    output = nw.get_noise_from_video(
        video,
        remove_background=False, #Set this to True to matte the foreground - and force the background to have no flow
        visualize=True,          #Generates nice visualization videos and previews in Jupyter notebook
        save_files=True,         #Set this to False if you just want the noises without saving to a numpy file
        
        noise_channels=16,
        output_folder=noise_folder,
        resize_frames=FRAME,
        resize_flow=FLOW,
        downscale_factor=round(FRAME * FLOW) * LATENT,
    )
    
    print("Noise shape:"  ,output.numpy_noises.shape)
    print("Flow shape:"   ,output.numpy_flows .shape)
    print("Output folder:",output.output_folder)

do_noisewarp()

In [ ]:
while True:
    try:
        load_random_video()
        do_noisewarp()
    except Exception:
        print_stack_trace()